# CNN baseline LRG — split estratificado (diagnostico)

**Objetivo**: descobrir se o NMAD do `cnn_baseline` original (0.0038) esta **otimista por causa do split aleatorio**.

O baseline original usa `make_or_load_split` (replica `train_test_split` aleatorio). O `cnn_optuna_flex` usa split **estratificado por z** e deu NMAD = 0.0088 no test. Os dois Optuna (simples 0.0081, flex 0.0088) concordam em ~0.008, o que levanta a suspeita de que o split aleatorio do baseline vaza espectros quase-duplicados entre train e test.

**Experimento controlado**: treina a **MESMA arquitetura do baseline** (`PaddedSpectralCNN`, lr=3e-4, 50 epochs, batch 64), mas no **MESMO test set do flex** (carrega o `split_idx.npz` do flex). Assim isolamos a unica variavel: o split.

Leitura do resultado:
- Se NMAD ~ 0.0088 (igual ao flex) -> o **split aleatorio estava inflando** o baseline. O numero honesto do LRG e' ~0.008.
- Se NMAD ~ 0.0038 (igual ao baseline original) -> a **arquitetura do baseline e' genuinamente melhor** que a escolhida pelo Optuna flex; nao e' problema de split.

## 1. Parametros

In [ ]:
OBJECT_TYPE   = 'LRG'
SEED          = 42
EPOCHS        = 50
BATCH_SIZE    = 64
LEARNING_RATE = 3e-4
# split_idx.npz do run flex (mesmo test set). Caminho relativo a RESULTS_DIR/OBJECT_TYPE.
FLEX_SPLIT_REL = 'cnn_optuna_flex/flex-LRG-t50_e40_s42-mae_nozw-4e97b2e3/split_idx.npz'

## 2. Setup

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR
while not (PROJECT_ROOT / 'config.py').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

from config import paths_for, RESULTS_DIR as PROJECT_RESULTS, MODELS_DIR, SPLITS_DIR, print_info
from src.data import load_spectral_dataset, normalize_spectra, make_or_load_split
from src.models import PaddedSpectralCNN
from src.evaluation import compute_redshift_metrics, metrics_by_redshift_bin

print_info()

paths = paths_for(OBJECT_TYPE)
HDF5_PATH = paths['spectra_h5'].with_name(f'{OBJECT_TYPE}spectra_padded.h5')

RESULTS_DIR = PROJECT_RESULTS / OBJECT_TYPE / 'cnn_baseline_stratified'
MODEL_DIR   = MODELS_DIR / OBJECT_TYPE / 'cnn_baseline_stratified'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FLEX_SPLIT = PROJECT_RESULTS / OBJECT_TYPE / FLEX_SPLIT_REL

sns.set_theme(style='whitegrid', palette='deep')
print(f'\nObjeto     : {OBJECT_TYPE}')
print(f'HDF5       : {HDF5_PATH}')
print(f'Flex split : {FLEX_SPLIT}')

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs visiveis: {len(gpus)}')

## 3. Dados — usa o MESMO split do flex

Carrega o `split_idx.npz` do flex e reconstroi os indices no dataset completo:
- `test  = test_idx`
- `train = pool_idx[train_in_pool]`
- `val   = pool_idx[val_in_pool]`

Isso garante **exatamente as mesmas galaxias** em cada split que o flex usou.

In [ ]:
X, y, n_wave = load_spectral_dataset(HDF5_PATH)
X = normalize_spectra(X).astype(np.float32)
y = y.astype(np.float32)
print(f'X: {X.shape}  y: {y.shape}  n_wave: {n_wave}')

# Split canonico ESTRATIFICADO por z (splits/<OBJ>_split.npz) — mesmo test set
# de TODOS os modelos. Substitui a leitura do split_idx.npz de um run flex.
train_idx, val_idx, test_idx = make_or_load_split(OBJECT_TYPE, y, SPLITS_DIR)

# Sanity: sem overlap entre os 3 splits
assert len(set(train_idx) & set(test_idx)) == 0, 'LEAK train/test!'
assert len(set(val_idx)   & set(test_idx)) == 0, 'LEAK val/test!'
assert len(set(train_idx) & set(val_idx))  == 0, 'LEAK train/val!'

X_train, y_train = X[train_idx], y[train_idx]
X_val,   y_val   = X[val_idx],   y[val_idx]
X_test,  y_test  = X[test_idx],  y[test_idx]

print(f'\nTrain: {len(y_train):,}  z_mean={y_train.mean():.4f}')
print(f'Val  : {len(y_val):,}  z_mean={y_val.mean():.4f}')
print(f'Test : {len(y_test):,}  z_mean={y_test.mean():.4f}')
print(f'\n(mesmas {len(y_test):,} galaxias de test que o flex)')

## 4. Treinar — MESMA arquitetura do baseline original

`PaddedSpectralCNN` com os HPs identicos ao `cnn_lrg.ipynb` (lr=3e-4, 50 epochs, batch 64). So' o split mudou.

In [ ]:
cnn = PaddedSpectralCNN(n_wave=n_wave, learning_rate=LEARNING_RATE)
cnn.build()
cnn.model.summary()

cnn.fit(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    patience_es=20, patience_lr=10,
    verbose=2,
)

results = compute_redshift_metrics(y_test, cnn.predict(X_test))
print(f"\n{'='*55}")
print(f'RESULTADO TEST estratificado (n={len(y_test):,}, train_time={cnn.train_time:.0f}s)')
print(f"{'='*55}")
print(f"  RMSE       : {results['rmse']:.4f}")
print(f"  MAE        : {results['mae']:.4f}")
print(f"  R2         : {results['r2']:.4f}")
print(f"  bias       : {results['bias']:+.4e}")
print(f"  sigma_NMAD : {results['nmad']:.4e}")
print(f"  outliers >0.05 : {results['outliers_0.05_pct']:.2f}%")
print(f"  outliers >0.15 : {results['outliers_0.15_pct']:.2f}%")

## 5. Comparacao — a pergunta central

Mesmo conjunto de test (estratificado) pra baseline-arch (este nb) vs flex. E o baseline original (split aleatorio) como referencia do numero "suspeito".

In [ ]:
nmad_this = results['nmad']

print('=' * 64)
print('COMPARACAO (LRG)')
print('=' * 64)
print(f'  baseline original (split ALEATORIO)        : sigma_NMAD = 0.0038')
print(f'  CNN flex Optuna  (split ESTRATIFICADO)     : sigma_NMAD = 0.0088')
print(f'  ESTE: baseline-arch (split ESTRATIFICADO)  : sigma_NMAD = {nmad_this:.4f}')
print('=' * 64)

if nmad_this > 0.006:
    print('\n>> CONCLUSAO: no MESMO split estratificado, a arquitetura do baseline')
    print('   tambem piora (~0.008). Logo o 0.0038 do baseline original estava')
    print('   OTIMISTA por causa do split aleatorio (provavel leak de quase-duplicatas).')
    print('   Numero honesto do LRG fica em ~0.008.')
else:
    print('\n>> CONCLUSAO: a arquitetura do baseline mantem ~0.0038 mesmo no split')
    print('   estratificado. Logo NAO eh problema de split: a arquitetura do baseline')
    print('   supera a escolhida pelo Optuna flex. Investigar o flex (espaco de busca,')
    print('   epochs do refit, overfitting a val no HPO).')

## 6. Salvar

In [ ]:
cnn.save(MODEL_DIR / 'cnn_baseline_stratified.keras')

# compute_redshift_metrics devolve so' metricas escalares; recomputa os arrays aqui
y_pred_test  = np.asarray(cnn.predict(X_test)).ravel()
delta_z_test = (y_pred_test - y_test) / (1.0 + y_test)

np.savez(
    RESULTS_DIR / 'predictions.npz',
    y_test=y_test, y_pred=y_pred_test,
    delta_z=delta_z_test, test_idx=test_idx,
)

scalar_keys = ['rmse', 'mae', 'r2', 'bias', 'nmad',
               'outliers_pct', 'outliers_0.05_pct', 'outliers_0.15_pct']
row = {'model': 'cnn_baseline_stratified', 'object': OBJECT_TYPE,
       'split': 'stratified_from_flex',
       'n_test': len(y_test), 'train_time_s': cnn.train_time,
       **{k: results[k] for k in scalar_keys}}
pd.DataFrame([row]).to_csv(RESULTS_DIR / 'metrics.csv', index=False)
print(f'Salvo:\n  {MODEL_DIR / "cnn_baseline_stratified.keras"}\n  {RESULTS_DIR / "metrics.csv"}')